In [ ]:
# Set up Step 6 for pilot_full.
#
# Upload the config file, read Step 5 .pt files from cfg.STEP6_INPUT_DIR,
# and prepare the configured Step 6 input/output directories.

import os
import sys
import shutil
import importlib
from pathlib import Path

from google.colab import files
from google.colab import drive

drive.mount("/content/drive")

PILOT_ROOT = "/content/pilot_full"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

print("Upload current full config file")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.endswith(".py") and "config" in name:
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)

import pilot_full.config as cfg
importlib.reload(cfg)

print("\nLoaded config:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("DATA_DIR:", cfg.DATA_DIR)
print("STEP6_ACTIVE_EXPERIMENT_ID:", cfg.STEP6_ACTIVE_EXPERIMENT_ID)
print("STEP6_EXPERIMENT_NAME:", cfg.STEP6_EXPERIMENT_NAME)
print("STEP6_SAMPLE_FAMILY:", cfg.STEP6_SAMPLE_FAMILY)
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)
print("STEP6_KEEP_EVIDENCE_TYPES:", cfg.STEP6_KEEP_EVIDENCE_TYPES)
print("STEP6_FEATURE_KEY:", cfg.STEP6_FEATURE_KEY)
print("STEP6_LABEL_FIELD:", cfg.STEP6_LABEL_FIELD)

if cfg.STEP6_INPUT_DIR is None:
    raise ValueError(
        "cfg.STEP6_INPUT_DIR is None. "
        "This Step6 notebook expects a single-family experiment."
    )

input_dir = Path(cfg.STEP6_INPUT_DIR)

if not input_dir.exists():
    raise FileNotFoundError(f"STEP6_INPUT_DIR does not exist: {input_dir}")

pt_files = sorted(input_dir.rglob("*.pt"))

if not pt_files:
    raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {input_dir}")

if os.path.exists(cfg.STEP6_OUTPUT_DIR):
    shutil.rmtree(cfg.STEP6_OUTPUT_DIR)

os.makedirs(cfg.STEP6_OUTPUT_DIR, exist_ok=True)

print("\nSetup complete.")
print("Config copied to:", config_target_path)
print("Number of .pt files found:", len(pt_files))
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)

for p in pt_files[:10]:
    print("-", p)

Mounted at /content/drive
Upload current full config file


Saving config_positional_context_full_drive.py to config_positional_context_full_drive.py

Loaded config:
PILOT_ROOT: /content/pilot_full
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data
STEP6_ACTIVE_EXPERIMENT_ID: composition_implicit_transitive_relation_ld
STEP6_EXPERIMENT_NAME: step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split
STEP6_SAMPLE_FAMILY: composition
STEP6_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_composition_qwen2_5_3b
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split
STEP6_KEEP_EVIDENCE_TYPES: {'implicit_transitive'}
STEP6_FEATURE_KEY: layer_diff_features
STEP6_LABEL_FIELD: relation

Setup complete.
Config copied to: /content/pilot_full/config.py
Number of .

In [ ]:
import shutil
import time
from pathlib import Path

USE_LOCAL_CONTENT_CACHE = getattr(cfg, "STEP6_USE_LOCAL_CONTENT_CACHE", True)
LOCAL_CONTENT_CACHE_ROOT = getattr(
    cfg,
    "STEP6_LOCAL_CONTENT_CACHE_ROOT",
    f"/content/step5_hs_cache_{cfg.STEP6_MODEL_TAG}_{cfg.STEP6_EXPERIMENT_SHORT_NAME}",
)
LOCAL_CACHE_SAFETY_MARGIN_GB = getattr(
    cfg,
    "STEP6_LOCAL_CACHE_SAFETY_MARGIN_GB",
    20.0,
)

drive_input_dir = Path(cfg.STEP6_INPUT_DIR)
local_input_dir = Path(LOCAL_CONTENT_CACHE_ROOT) / cfg.STEP6_SAMPLE_FAMILY

if not drive_input_dir.exists():
    raise FileNotFoundError(f"Drive Step5 input dir does not exist: {drive_input_dir}")

drive_pt_files = sorted(drive_input_dir.rglob("*.pt"))

if not drive_pt_files:
    raise FileNotFoundError(f"No .pt files found in Drive input dir: {drive_input_dir}")

print("Drive Step5 input dir:", drive_input_dir)
print("Number of Drive .pt files:", len(drive_pt_files))

if USE_LOCAL_CONTENT_CACHE:
    total_size_bytes = sum(p.stat().st_size for p in drive_pt_files)
    total_size_gb = total_size_bytes / 1024**3

    free_gb = shutil.disk_usage("/content").free / 1024**3

    print("\nLocal cache pre-check")
    print("Total Step5 .pt size: %.2f GB" % total_size_gb)
    print("Free /content disk: %.2f GB" % free_gb)
    print("Safety margin: %.2f GB" % LOCAL_CACHE_SAFETY_MARGIN_GB)
    print("Local cache dir:", local_input_dir)

    if total_size_gb + LOCAL_CACHE_SAFETY_MARGIN_GB > free_gb:
        raise RuntimeError(
            "Not enough /content disk space for local cache.\n"
            f"Total Step5 .pt size: {total_size_gb:.2f} GB\n"
            f"Free /content disk: {free_gb:.2f} GB\n"
            f"Safety margin: {LOCAL_CACHE_SAFETY_MARGIN_GB:.2f} GB\n"
            "Set STEP6_USE_LOCAL_CONTENT_CACHE=False or run a smaller subset."
        )

    local_input_dir.mkdir(parents=True, exist_ok=True)

    start_time = time.time()
    copied = 0
    reused = 0

    for i, src in enumerate(drive_pt_files, start=1):
        rel = src.relative_to(drive_input_dir)
        dst = local_input_dir / rel

        dst.parent.mkdir(parents=True, exist_ok=True)

        src_size = src.stat().st_size

        if dst.exists() and dst.stat().st_size == src_size:
            reused += 1
            print(f"[{i}/{len(drive_pt_files)}] reused: {rel}")
            continue

        tmp = Path(str(dst) + ".tmp")

        if tmp.exists():
            tmp.unlink()

        print(f"[{i}/{len(drive_pt_files)}] copying: {rel}")
        shutil.copyfile(src, tmp)

        copied_size = tmp.stat().st_size

        if copied_size != src_size:
            raise IOError(
                f"Copied file size mismatch for {rel}: "
                f"copied={copied_size}, expected={src_size}"
            )

        os.replace(tmp, dst)
        copied += 1

    elapsed = time.time() - start_time

    STEP6_INPUT_DIR = str(local_input_dir)

    print("\nLocal cache ready.")
    print("Copied files:", copied)
    print("Reused files:", reused)
    print("Elapsed: %.2f min" % (elapsed / 60))
    print("Active STEP6_INPUT_DIR:", STEP6_INPUT_DIR)

else:
    STEP6_INPUT_DIR = str(drive_input_dir)

    print("\nUSE_LOCAL_CONTENT_CACHE=False")
    print("Active STEP6_INPUT_DIR:", STEP6_INPUT_DIR)

Drive Step5 input dir: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_composition_qwen2_5_3b
Number of Drive .pt files: 94

Local cache pre-check
Total Step5 .pt size: 1.13 GB
Free /content disk: 188.60 GB
Safety margin: 20.00 GB
Local cache dir: /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition
[1/94] copying: FloorPlan10_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[2/94] copying: FloorPlan11_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[3/94] copying: FloorPlan13_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[4/94] copying: FloorPlan14_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[5/94] copying: FloorPlan15_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[6/94] copying: FloorPlan16_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[7/94] copying: FloorPlan17_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
[8/94] copying: FloorPlan18_step4_composition_samp

In [ ]:
# Import dependencies and load the Step 6 config.
#
# This run probes implicit-transitive spatial relations from Step 5 hidden states:
# X = hidden(subject) - hidden(object), y = relation.

import os
import sys
import json
import random
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

# Basic config

EXPERIMENT_NAME = cfg.STEP6_EXPERIMENT_NAME
RANDOM_SEED = getattr(cfg, "RANDOM_SEED", 42)

MODEL_NAME = getattr(cfg, "STEP6_MODEL_NAME", cfg.STEP5_MODEL_NAME)
MODEL_TAG = getattr(cfg, "STEP6_MODEL_TAG", cfg.STEP5_MODEL_TAG)

STEP6_INPUT_DIR = globals().get("STEP6_INPUT_DIR", cfg.STEP6_INPUT_DIR)
STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR

FEATURE_KEY = cfg.STEP6_FEATURE_KEY
LABEL_FIELD = cfg.STEP6_LABEL_FIELD

TRAIN_SCENES = list(cfg.STEP6_TRAIN_SCENES)
TEST_SCENES = list(cfg.STEP6_TEST_SCENES)

REQUIRE_EXPLICIT_SCENE_SPLIT = getattr(
    cfg,
    "STEP6_REQUIRE_EXPLICIT_SCENE_SPLIT",
    True,
)

LOGREG_MAX_ITER = getattr(cfg, "STEP6_LOGREG_MAX_ITER", 5000)
LOGREG_C = getattr(cfg, "STEP6_LOGREG_C", 1.0)
LOGREG_CLASS_WEIGHT = getattr(cfg, "STEP6_LOGREG_CLASS_WEIGHT", "balanced")
LOGREG_SOLVER = getattr(cfg, "STEP6_LOGREG_SOLVER", "lbfgs")

PRINT_DATASET_SUMMARY = getattr(cfg, "STEP6_PRINT_DATASET_SUMMARY", True)
PRINT_LAYER_PROGRESS = getattr(cfg, "STEP6_PRINT_LAYER_PROGRESS", True)
PRINT_TOP_LAYERS = getattr(cfg, "STEP6_PRINT_TOP_LAYERS", True)
NUM_TOP_LAYERS_TO_PRINT = getattr(cfg, "STEP6_NUM_TOP_LAYERS_TO_PRINT", 10)

# Filtering from active Step6 experiment
ALLOWED_EVIDENCE_TYPES = set(cfg.STEP6_KEEP_EVIDENCE_TYPES)

ALLOWED_LABELS = set(getattr(
    cfg,
    "STEP6_ALLOWED_LABELS",
    {"left_of", "right_of", "above", "below"},
))

DROP_NONE_LABEL = getattr(cfg, "STEP6_DROP_NONE_LABEL", True)
DROP_EMPTY_LABEL = getattr(cfg, "STEP6_DROP_EMPTY_LABEL", True)

# Only meaningful for composition / implicit-transitive samples.
GROUP_BY_IMPLICIT_RULE = getattr(
    cfg,
    "STEP6_GROUP_BY_IMPLICIT_RULE",
    True,
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

print("Step 6 config loaded.")
print("STEP6_ACTIVE_EXPERIMENT_ID:", cfg.STEP6_ACTIVE_EXPERIMENT_ID)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("TRAIN_SCENES:", TRAIN_SCENES)
print("TEST_SCENES:", TEST_SCENES)

print("\nCurrent Step6 task:")
print("Decode implicit relation from hidden-state features.")
print("ALLOWED_EVIDENCE_TYPES:", sorted(ALLOWED_EVIDENCE_TYPES))
print("ALLOWED_LABELS:", sorted(ALLOWED_LABELS))
print("DROP_NONE_LABEL:", DROP_NONE_LABEL)
print("DROP_EMPTY_LABEL:", DROP_EMPTY_LABEL)
print("GROUP_BY_IMPLICIT_RULE:", GROUP_BY_IMPLICIT_RULE)

Step 6 config loaded.
STEP6_ACTIVE_EXPERIMENT_ID: composition_implicit_transitive_relation_ld
EXPERIMENT_NAME: step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split
MODEL_NAME: Qwen/Qwen2.5-3B
MODEL_TAG: qwen2_5_3b
STEP6_INPUT_DIR: /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition
STEP6_OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split
FEATURE_KEY: layer_diff_features
LABEL_FIELD: relation
TRAIN_SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6', 'FloorPlan7', 'FloorPlan8', 'FloorPlan9', 'FloorPlan10', 'FloorPlan11', 'FloorPlan12', 'FloorPlan13', 'FloorPlan14', 'FloorPlan15', 'FloorPlan16', 'FloorPlan17', 'FloorPlan18', 'FloorPlan19', 'FloorPlan20', 'FloorPlan21', 'FloorPlan22', 'FloorPlan23', 'FloorPlan24', 'FloorPlan25', 'FloorPl

In [ ]:
# Helper functions for loading Step 5 records

def discover_step5_pt_files(input_dir: str) -> List[str]:
    pt_paths = sorted(Path(input_dir).rglob("*.pt"))

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {input_dir}")

    return [str(p) for p in pt_paths]


def normalize_record(raw_record: Dict[str, Any], source_pt_file: str) -> Dict[str, Any]:
    """
    Normalize one Step 5 record into a predictable structure.

    Expected current Step5 fields:
      - layer_diff_features / layer_concat_features / ...
      - relation
      - scene
      - evidence_type = implicit_transitive
      - implicit_rule
      - anchor_uid
      - subject_alias / object_alias
    """

    rec = dict(raw_record)
    rec["_source_pt_file"] = source_pt_file

    # Some versions save metadata as rec["metadata"], while others flatten it.
    metadata = rec.get("metadata", {})

    if metadata is None:
        metadata = {}

    if not isinstance(metadata, dict):
        metadata = {}

    for k, v in metadata.items():
        rec.setdefault(k, v)

    evidence = rec.get("evidence", {})
    if isinstance(evidence, dict):
        rec.setdefault("evidence_type", evidence.get("evidence_type"))
        rec.setdefault(
            "probe_direction_relative_to_text",
            evidence.get("probe_direction_relative_to_text"),
        )
        rec.setdefault("implicit_rule", evidence.get("implicit_rule"))
        rec.setdefault("anchor_uid", evidence.get("anchor_uid"))
        rec.setdefault("supporting_relations", evidence.get("supporting_relations"))
        rec.setdefault("num_supporting_paths", evidence.get("num_supporting_paths"))

    probe_pair = rec.get("probe_pair", {})
    if isinstance(probe_pair, dict):
        rec.setdefault("probe_subject_uid", probe_pair.get("probe_subject_uid"))
        rec.setdefault("probe_object_uid", probe_pair.get("probe_object_uid"))
        rec.setdefault("probe_relation_label", probe_pair.get("probe_relation_label"))
        rec.setdefault("is_implicit_transitive", probe_pair.get("is_implicit_transitive"))
        rec.setdefault("is_explicit_in_text", probe_pair.get("is_explicit_in_text"))

    # Standardize relation label.
    if LABEL_FIELD not in rec:
        if "relation" in rec:
            rec[LABEL_FIELD] = rec["relation"]
        elif "probe_relation_label" in rec:
            rec[LABEL_FIELD] = rec["probe_relation_label"]
        elif isinstance(probe_pair, dict) and "probe_relation_label" in probe_pair:
            rec[LABEL_FIELD] = probe_pair["probe_relation_label"]
        else:
            raise KeyError(f"Could not find label field '{LABEL_FIELD}' in record.")

    # Standardize scene.
    if "scene" not in rec:
        if "source_step3_scene" in rec:
            rec["scene"] = rec["source_step3_scene"]
        else:
            raise KeyError("Could not find scene field in record.")

    # Standardize subject/object aliases if needed.
    if "subject_alias" not in rec:
        rec["subject_alias"] = (
            rec.get("probe_subject_uid")
            or rec.get("subject_uid")
            or rec.get("subject")
        )

    if "object_alias" not in rec:
        rec["object_alias"] = (
            rec.get("probe_object_uid")
            or rec.get("object_uid")
            or rec.get("object")
        )

    return rec


def extract_records_from_payload(payload: Any, source_pt_file: str) -> List[Dict[str, Any]]:
    """
    Supports several possible Step 5 payload formats:
      - list[record]
      - dict with key "records"
      - dict with key "data"
      - dict with key "examples"
    """

    if isinstance(payload, list):
        raw_records = payload
    elif isinstance(payload, dict):
        if "records" in payload:
            raw_records = payload["records"]
        elif "data" in payload:
            raw_records = payload["data"]
        elif "examples" in payload:
            raw_records = payload["examples"]
        else:
            raise KeyError(
                f"Unsupported .pt payload keys in {source_pt_file}: {list(payload.keys())}"
            )
    else:
        raise TypeError(f"Unsupported .pt payload type in {source_pt_file}: {type(payload)}")

    records = [
        normalize_record(r, source_pt_file=source_pt_file)
        for r in raw_records
    ]

    return records


def to_numpy_feature(x: Any) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float().numpy()

    arr = np.asarray(x)

    if arr.dtype == np.dtype("O"):
        arr = arr.astype(np.float32)

    return arr.astype(np.float32, copy=False)


print("Helper functions loaded.")

Helper functions loaded.


In [ ]:
# Load Step 5 records, filter target samples, and build feature tensor

pt_files = discover_step5_pt_files(STEP6_INPUT_DIR)

all_records_raw = []

for i, pt_path in enumerate(pt_files, start=1):
    print(f"[load {i}/{len(pt_files)}] {pt_path}")

    payload = torch.load(pt_path, map_location="cpu")
    records = extract_records_from_payload(payload, source_pt_file=pt_path)

    print(f"  records loaded: {len(records)}")
    all_records_raw.extend(records)

if not all_records_raw:
    raise ValueError("No Step 5 records loaded.")

print("Number of raw Step 5 records loaded:", len(all_records_raw))
print("Number of source .pt files:", len(pt_files))

# Filter to current Step6 target samples

filtered_records = []

for r in all_records_raw:
    evidence_type = r.get("evidence_type")
    label = r.get(LABEL_FIELD)

    if evidence_type not in ALLOWED_EVIDENCE_TYPES:
        continue

    if DROP_EMPTY_LABEL and (label is None or label == ""):
        continue

    if DROP_NONE_LABEL and label == getattr(cfg, "NONE_RELATION_LABEL", "none"):
        continue

    if label not in ALLOWED_LABELS:
        continue

    filtered_records.append(r)

if not filtered_records:
    raise ValueError(
        "No records left after filtering. "
        f"ALLOWED_EVIDENCE_TYPES={ALLOWED_EVIDENCE_TYPES}, "
        f"ALLOWED_LABELS={ALLOWED_LABELS}, "
        f"DROP_NONE_LABEL={DROP_NONE_LABEL}, "
        f"DROP_EMPTY_LABEL={DROP_EMPTY_LABEL}"
    )

all_records = filtered_records

print("\nNumber of records after filtering:", len(all_records))


missing_feature = [i for i, r in enumerate(all_records) if FEATURE_KEY not in r]
missing_label = [i for i, r in enumerate(all_records) if LABEL_FIELD not in r]

if missing_feature:
    raise KeyError(f"{len(missing_feature)} records are missing feature key: {FEATURE_KEY}")

if missing_label:
    raise KeyError(f"{len(missing_label)} records are missing label field: {LABEL_FIELD}")

# Stack features
# Expected shape per record: [num_layers, hidden_dim]

features = []

for r in all_records:
    f = to_numpy_feature(r[FEATURE_KEY])

    if f.ndim != 2:
        raise ValueError(
            f"Expected feature shape [num_layers, dim], got {f.shape} "
            f"for sample_id={r.get('sample_id')}"
        )

    features.append(f)

X_all = np.stack(features, axis=0)  # [num_samples, num_layers, dim]
y_all = np.array([r[LABEL_FIELD] for r in all_records])
metadata = all_records

num_samples, num_layers, feature_dim = X_all.shape

print("\nX_all shape:", X_all.shape)
print("num_samples:", num_samples)
print("num_layers:", num_layers)
print("feature_dim:", feature_dim)

print("\nLabel counts:")
print(dict(Counter(y_all)))

print("\nScene counts:")
print(dict(Counter(r["scene"] for r in metadata)))

print("\nEvidence type counts:")
print(dict(Counter(r.get("evidence_type", "unknown") for r in metadata)))

print("\nImplicit rule counts:")
print(dict(Counter(r.get("implicit_rule", "unknown") for r in metadata)))

print("\nProbe direction counts:")
print(dict(Counter(r.get("probe_direction_relative_to_text", "unknown") for r in metadata)))

[load 1/94] /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition/FloorPlan10_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
  records loaded: 7
[load 2/94] /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition/FloorPlan11_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
  records loaded: 37
[load 3/94] /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition/FloorPlan13_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
  records loaded: 31
[load 4/94] /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition/FloorPlan14_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
  records loaded: 45
[load 5/94] /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition/FloorPlan15_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
  records loaded: 18
[load 6/94] /content/step5_hs_cache_qwen2_5_3b_comp_itr_rel_ld/composition/FloorPlan16_step4_composition_samples_diverse_step5_hs_qwen2_5_3b.pt
  records loaded: 6
[load 7/94] 

In [ ]:
# Build the scene split and filter to common implicit-transitive labels.
#
# Use TRAIN_SCENES for training and TEST_SCENES for testing. The remaining
# labels are usually {left_of, right_of, above, below}.

all_scenes = sorted(set(r["scene"] for r in metadata))

missing_train = sorted(set(TRAIN_SCENES) - set(all_scenes))
missing_test = sorted(set(TEST_SCENES) - set(all_scenes))

present_train_scenes = sorted(set(TRAIN_SCENES) & set(all_scenes))
present_test_scenes = sorted(set(TEST_SCENES) & set(all_scenes))

if REQUIRE_EXPLICIT_SCENE_SPLIT:
    print("Scene split availability check:")
    print("  available scenes:", len(all_scenes))
    print("  present train scenes:", len(present_train_scenes), "/", len(TRAIN_SCENES))
    print("  present test scenes:", len(present_test_scenes), "/", len(TEST_SCENES))
    print("  missing train scenes:", missing_train)
    print("  missing test scenes:", missing_test)

    if len(present_train_scenes) == 0:
        raise ValueError(
            "No configured training scene has any sample after filtering. "
            f"missing_train={missing_train}, available_scenes={all_scenes}"
        )

    if len(present_test_scenes) == 0:
        raise ValueError(
            "No configured test scene has any sample after filtering. "
            f"missing_test={missing_test}, available_scenes={all_scenes}"
        )

train_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TRAIN_SCENES],
    dtype=np.int64,
)

test_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TEST_SCENES],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples selected.")

if len(test_idx) == 0:
    raise ValueError("No test examples selected.")

train_labels_before = set(y_all[train_idx])
test_labels_before = set(y_all[test_idx])
common_labels = sorted(train_labels_before & test_labels_before)

if len(common_labels) < 2:
    raise ValueError(
        f"Need at least two common labels for classification. "
        f"train_labels={sorted(train_labels_before)}, "
        f"test_labels={sorted(test_labels_before)}, "
        f"common_labels={common_labels}"
    )

removed_train_only_labels = sorted(train_labels_before - test_labels_before)
removed_test_only_labels = sorted(test_labels_before - train_labels_before)

train_idx = np.array(
    [idx for idx in train_idx if y_all[idx] in common_labels],
    dtype=np.int64,
)

test_idx = np.array(
    [idx for idx in test_idx if y_all[idx] in common_labels],
    dtype=np.int64,
)

label_order = common_labels

scene_split_info = {
    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "available_scenes": all_scenes,

    "missing_train_scenes": missing_train,
    "missing_test_scenes": missing_test,
    "present_train_scenes": present_train_scenes,
    "present_test_scenes": present_test_scenes,

    "num_train_before_common_label_filter": int(sum(r["scene"] in TRAIN_SCENES for r in metadata)),
    "num_test_before_common_label_filter": int(sum(r["scene"] in TEST_SCENES for r in metadata)),

    "train_labels_before": sorted(train_labels_before),
    "test_labels_before": sorted(test_labels_before),
    "common_labels": common_labels,
    "removed_train_only_labels": removed_train_only_labels,
    "removed_test_only_labels": removed_test_only_labels,

    "num_train_final": int(len(train_idx)),
    "num_test_final": int(len(test_idx)),

    "train_label_counts_final": dict(Counter(y_all[train_idx])),
    "test_label_counts_final": dict(Counter(y_all[test_idx])),

    "train_scene_counts_final": dict(Counter(metadata[int(idx)]["scene"] for idx in train_idx)),
    "test_scene_counts_final": dict(Counter(metadata[int(idx)]["scene"] for idx in test_idx)),

    "train_evidence_type_counts_final": dict(
        Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in train_idx)
    ),
    "test_evidence_type_counts_final": dict(
        Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in test_idx)
    ),

    "train_implicit_rule_counts_final": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)
    ),
    "test_implicit_rule_counts_final": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)
    ),
}

print("Scene split complete.")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

print("\nFinal label_order:")
print(label_order)

print("\nFinal train label counts:")
print(dict(Counter(y_all[train_idx])))

print("\nFinal test label counts:")
print(dict(Counter(y_all[test_idx])))

print("\nFinal train implicit rule counts:")
print(dict(Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)))

print("\nFinal test implicit rule counts:")
print(dict(Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)))

Scene split availability check:
  available scenes: 94
  present train scenes: 77 / 100
  present test scenes: 17 / 20
  missing train scenes: ['FloorPlan12', 'FloorPlan202', 'FloorPlan210', 'FloorPlan211', 'FloorPlan212', 'FloorPlan213', 'FloorPlan214', 'FloorPlan217', 'FloorPlan222', 'FloorPlan223', 'FloorPlan224', 'FloorPlan312', 'FloorPlan315', 'FloorPlan318', 'FloorPlan322', 'FloorPlan401', 'FloorPlan405', 'FloorPlan411', 'FloorPlan413', 'FloorPlan415', 'FloorPlan417', 'FloorPlan421', 'FloorPlan424']
  missing test scenes: ['FloorPlan226', 'FloorPlan228', 'FloorPlan428']
Scene split complete.
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4",
    "FloorPlan5",
    "FloorPlan6",
    "FloorPlan7",
    "FloorPlan8",
    "FloorPlan9",
    "FloorPlan10",
    "FloorPlan11",
    "FloorPlan12",
    "FloorPlan13",
    "FloorPlan14",
    "FloorPlan15",
    "FloorPlan16",
    "FloorPlan17",
    "FloorPlan18",
    "FloorPlan19",
    "FloorPlan20",
  

In [ ]:
# Direction filtering is disabled for implicit-transitive samples.
#
# The A-C target relation is not stated directly in the text; the probe tests
# whether it can be decoded from the hidden states.

direction_filter_info = {
    "enabled": False,
    "reason": (
        "Current Step6 uses composition implicit-transitive samples. "
        "The target A-C relation is not directly stated in text, so the "
        "old explicit direct/inverse train-test direction protocol is disabled."
    ),
}

print("Direction filtering disabled for current implicit-transitive Step6.")
print(json.dumps(direction_filter_info, indent=2, ensure_ascii=False))

Direction filtering disabled for current implicit-transitive Step6.
{
  "enabled": false,
  "reason": "Current Step6 uses composition implicit-transitive samples. The target A-C relation is not directly stated in text, so the old explicit direct/inverse train-test direction protocol is disabled."
}


In [ ]:
final_split_summary = {
    "num_samples_loaded_after_filtering": int(num_samples),
    "num_train": int(len(train_idx)),
    "num_test": int(len(test_idx)),
    "label_order": label_order,

    "train_label_counts": dict(Counter(y_all[train_idx])),
    "test_label_counts": dict(Counter(y_all[test_idx])),

    "train_scene_counts": dict(Counter(metadata[int(idx)]["scene"] for idx in train_idx)),
    "test_scene_counts": dict(Counter(metadata[int(idx)]["scene"] for idx in test_idx)),

    "train_implicit_rule_counts": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)
    ),
    "test_implicit_rule_counts": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)
    ),

    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
}

print("Final train/test split summary:")
print(json.dumps(final_split_summary, indent=2, ensure_ascii=False))

Final train/test split summary:
{
  "num_samples_loaded_after_filtering": 1577,
  "num_train": 1204,
  "num_test": 373,
  "label_order": [
    "above",
    "below",
    "left_of",
    "right_of"
  ],
  "train_label_counts": {
    "right_of": 578,
    "left_of": 587,
    "below": 17,
    "above": 22
  },
  "test_label_counts": {
    "left_of": 185,
    "right_of": 176,
    "above": 6,
    "below": 6
  },
  "train_scene_counts": {
    "FloorPlan10": 7,
    "FloorPlan11": 37,
    "FloorPlan13": 31,
    "FloorPlan14": 45,
    "FloorPlan15": 18,
    "FloorPlan16": 6,
    "FloorPlan17": 23,
    "FloorPlan18": 6,
    "FloorPlan19": 18,
    "FloorPlan1": 29,
    "FloorPlan201": 13,
    "FloorPlan203": 14,
    "FloorPlan204": 3,
    "FloorPlan205": 19,
    "FloorPlan206": 12,
    "FloorPlan207": 2,
    "FloorPlan208": 1,
    "FloorPlan209": 25,
    "FloorPlan20": 61,
    "FloorPlan215": 15,
    "FloorPlan216": 20,
    "FloorPlan218": 5,
    "FloorPlan219": 16,
    "FloorPlan21": 5,
    "FloorPl

In [ ]:
test_subset_indices = {
    "test_overall": test_idx,
}

has_implicit_rule = any(
    metadata[int(idx)].get("implicit_rule") is not None
    for idx in test_idx
)

if GROUP_BY_IMPLICIT_RULE and has_implicit_rule:
    test_rules = sorted(
        set(
            metadata[int(idx)].get("implicit_rule", "unknown")
            for idx in test_idx
        )
    )

    for rule in test_rules:
        subset_key = f"test_rule_{rule}"
        rule_idx = np.array(
            [
                idx for idx in test_idx
                if metadata[int(idx)].get("implicit_rule", "unknown") == rule
            ],
            dtype=np.int64,
        )
        test_subset_indices[subset_key] = rule_idx

test_subset_summary = {}

for subset_key, idxs in test_subset_indices.items():
    test_subset_summary[subset_key] = {
        "num_examples": int(len(idxs)),
        "label_counts": dict(Counter(y_all[idxs])) if len(idxs) else {},
        "evidence_type_counts": dict(
            Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in idxs)
        ) if len(idxs) else {},
        "implicit_rule_counts": dict(
            Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in idxs)
        ) if len(idxs) else {},
        "scene_counts": dict(
            Counter(metadata[int(idx)].get("scene", "unknown") for idx in idxs)
        ) if len(idxs) else {},
    }

print("Test subset summary:")
print(json.dumps(test_subset_summary, indent=2, ensure_ascii=False))

Test subset summary:
{
  "test_overall": {
    "num_examples": 373,
    "label_counts": {
      "left_of": 185,
      "right_of": 176,
      "above": 6,
      "below": 6
    },
    "evidence_type_counts": {
      "implicit_transitive": 373
    },
    "implicit_rule_counts": {
      "null": 373
    },
    "scene_counts": {
      "FloorPlan227": 13,
      "FloorPlan229": 2,
      "FloorPlan230": 8,
      "FloorPlan26": 39,
      "FloorPlan27": 25,
      "FloorPlan28": 35,
      "FloorPlan29": 67,
      "FloorPlan30": 30,
      "FloorPlan326": 4,
      "FloorPlan327": 16,
      "FloorPlan328": 28,
      "FloorPlan329": 7,
      "FloorPlan330": 11,
      "FloorPlan426": 8,
      "FloorPlan427": 53,
      "FloorPlan429": 13,
      "FloorPlan430": 14
    }
  }
}


In [ ]:
# Layer-wise training and evaluation

def evaluate_classifier(
    clf: LogisticRegression,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    labels: List[str],
) -> Dict[str, Any]:
    if len(y_eval) == 0:
        return {
            "num_examples": 0,
            "accuracy": None,
            "macro_f1": None,
            "label_order": labels,
            "classification_report": {},
            "confusion_matrix": [],
        }

    y_pred = clf.predict(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    macro_f1 = f1_score(
        y_eval,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    report = classification_report(
        y_eval,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_eval,
        y_pred,
        labels=labels,
    )

    return {
        "num_examples": int(len(y_eval)),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "label_order": labels,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


results_by_layer = []

for layer_id in range(num_layers):
    if PRINT_LAYER_PROGRESS:
        print(f"Training layer {layer_id}/{num_layers - 1}")

    X_train_layer = X_all[train_idx, layer_id, :]
    y_train_layer = y_all[train_idx]

    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        C=LOGREG_C,
        class_weight=LOGREG_CLASS_WEIGHT,
        solver=LOGREG_SOLVER,
        multi_class="auto",
        random_state=RANDOM_SEED,
    )

    clf.fit(X_train_layer, y_train_layer)

    layer_result = {
        "layer": int(layer_id),
        "train": evaluate_classifier(
            clf,
            X_train_layer,
            y_train_layer,
            labels=label_order,
        ),
    }

    for subset_key, idxs in test_subset_indices.items():
        X_eval = X_all[idxs, layer_id, :]
        y_eval = y_all[idxs]

        layer_result[subset_key] = evaluate_classifier(
            clf,
            X_eval,
            y_eval,
            labels=label_order,
        )

    # Current layer test performance

    overall_key = "test_overall"

    if overall_key not in layer_result:
        test_keys = [
            k for k in layer_result.keys()
            if k.startswith("test_")
        ]
        overall_key = test_keys[0] if test_keys else None

    if PRINT_LAYER_PROGRESS and overall_key is not None:
        overall_result = layer_result[overall_key]

        overall_acc = overall_result.get("accuracy")
        overall_macro_f1 = overall_result.get("macro_f1")

        if overall_acc is not None and overall_macro_f1 is not None:
            print(
                f"Layer {layer_id} done | "
                f"overall_macro_f1={overall_macro_f1:.4f} | "
                f"overall_acc={overall_acc:.4f}"
            )
        else:
            print(
                f"Layer {layer_id} done | "
                f"overall_macro_f1=None | "
                f"overall_acc=None"
            )

    results_by_layer.append(layer_result)

print("Layer-wise training and evaluation complete.")

Training layer 0/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 0 done | overall_macro_f1=0.2496 | overall_acc=0.3834
Training layer 1/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 1 done | overall_macro_f1=0.6618 | overall_acc=0.7936
Training layer 2/36


...

Layer 34 done | overall_macro_f1=0.3509 | overall_acc=0.5952
Training layer 35/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 35 done | overall_macro_f1=0.3670 | overall_acc=0.6193
Training layer 36/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer 36 done | overall_macro_f1=0.3637 | overall_acc=0.6113
Layer-wise training and evaluation complete.


In [ ]:
# Summarize top layers

summary_rows = []

test_subset_keys = list(test_subset_indices.keys())

for r in results_by_layer:
    row = {
        "layer": r["layer"],
        "train_accuracy": r["train"]["accuracy"],
        "train_macro_f1": r["train"]["macro_f1"],
    }

    for subset_key in test_subset_keys:
        row[f"{subset_key}_accuracy"] = r[subset_key]["accuracy"]
        row[f"{subset_key}_macro_f1"] = r[subset_key]["macro_f1"]
        row[f"{subset_key}_num_examples"] = r[subset_key]["num_examples"]

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_csv_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_layer_summary.csv",
)

summary_df.to_csv(summary_csv_path, index=False)

print("Saved layer summary CSV:")
print(summary_csv_path)

display(summary_df.head())

if PRINT_TOP_LAYERS:
    ranking_metrics = [
        "test_overall_accuracy",
        "test_overall_macro_f1",
    ]

    # Print rule-specific metrics
    for subset_key in test_subset_keys:
        if subset_key != "test_overall":
            ranking_metrics.append(f"{subset_key}_accuracy")

    for metric in ranking_metrics:
        if metric not in summary_df.columns:
            continue

        print("\n" + "=" * 80)
        print("Top layers by:", metric)
        print("=" * 80)

        cols = [
            "layer",
            metric,
            "test_overall_accuracy",
            "test_overall_macro_f1",
        ]

        # Add available subset accuracy columns.
        for subset_key in test_subset_keys:
            acc_col = f"{subset_key}_accuracy"
            if acc_col not in cols and acc_col in summary_df.columns:
                cols.append(acc_col)

        display(
            summary_df.sort_values(metric, ascending=False)
            .loc[:, cols]
            .head(NUM_TOP_LAYERS_TO_PRINT)
        )

Saved layer summary CSV:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split_qwen2_5_3b_layer_diff_features_layer_summary.csv


,layer,train_accuracy,train_macro_f1,test_overall_accuracy,test_overall_macro_f1,test_overall_num_examples
0,0,0.571429,0.401323,0.383378,0.249607,373
1,1,0.961794,0.916371,0.793566,0.661830,373
2,2,0.977575,0.963425,0.852547,0.718941,373
3,3,0.985880,0.979238,0.849866,0.729765,373
4,4,0.993355,0.989638,0.820375,0.594845,373



Top layers by: test_overall_accuracy


,layer,test_overall_accuracy,test_overall_accuracy,test_overall_macro_f1
2,2,0.852547,0.852547,0.718941
3,3,0.849866,0.849866,0.729765
5,5,0.847185,0.847185,0.686385
22,22,0.839142,0.839142,0.566507
19,19,0.831099,0.831099,0.783797
23,23,0.825737,0.825737,0.516660
21,21,0.825737,0.825737,0.597207
4,4,0.820375,0.820375,0.594845
26,26,0.817694,0.817694,0.565984
20,20,0.815013,0.815013,0.624607



Top layers by: test_overall_macro_f1


,layer,test_overall_macro_f1,test_overall_accuracy,test_overall_macro_f1
19,19,0.783797,0.831099,0.783797
17,17,0.750800,0.790885,0.750800
3,3,0.729765,0.849866,0.729765
2,2,0.718941,0.852547,0.718941
5,5,0.686385,0.847185,0.686385
1,1,0.661830,0.793566,0.661830
10,10,0.644923,0.726542,0.644923
18,18,0.641417,0.780161,0.641417
20,20,0.624607,0.815013,0.624607
7,7,0.614134,0.785523,0.614134


In [ ]:
output_payload = {
    "experiment_id": getattr(cfg, "STEP6_EXPERIMENT_ID", cfg.STEP6_ACTIVE_EXPERIMENT_ID),
    "experiment_name": EXPERIMENT_NAME,
    "experiment_description": getattr(cfg, "STEP6_EXPERIMENT_DESCRIPTION", ""),
    "script_family": getattr(cfg, "STEP6_SCRIPT_FAMILY", ""),

    "description": (
        "Layer-wise linear probing for implicit relation decoding. "
        "A logistic-regression probe is trained on Step5 hidden-state features "
        "to predict the spatial relation label between subject and object. "
        "The main feature is configured by cfg.STEP6_FEATURE_KEY. "
        "Evaluation uses an explicit scene split."
    ),

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,

    "probe_target": {
        "task": getattr(cfg, "STEP6_TARGET_TAG", "relation"),
        "input_feature": FEATURE_KEY,
        "label": LABEL_FIELD,
        "allowed_labels": sorted(ALLOWED_LABELS),
        "allowed_evidence_types": sorted(ALLOWED_EVIDENCE_TYPES),
        "feature_interpretation": "hidden(subject) - hidden(object) if FEATURE_KEY == layer_diff_features",
    },

    "num_samples_loaded_after_filtering": int(num_samples),
    "num_layers": int(num_layers),
    "feature_dim": int(feature_dim),
    "label_order": label_order,

    "scene_split_info": scene_split_info,
    "direction_filter_info": direction_filter_info,
    "final_split_summary": final_split_summary,
    "test_subset_summary": test_subset_summary,

    "step6_config": {
        "active_experiment_id": cfg.STEP6_ACTIVE_EXPERIMENT_ID,
        "experiment_name": EXPERIMENT_NAME,
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "sample_family": cfg.STEP6_SAMPLE_FAMILY,
        "feature_key": FEATURE_KEY,
        "label_field": LABEL_FIELD,
        "allowed_evidence_types": sorted(ALLOWED_EVIDENCE_TYPES),
        "allowed_labels": sorted(ALLOWED_LABELS),
        "train_scenes": TRAIN_SCENES,
        "test_scenes": TEST_SCENES,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_c": LOGREG_C,
        "logreg_class_weight": LOGREG_CLASS_WEIGHT,
        "logreg_solver": LOGREG_SOLVER,
        "random_seed": RANDOM_SEED,
        "group_by_implicit_rule": GROUP_BY_IMPLICIT_RULE,
    },

    "source_pt_files": pt_files,
    "results_by_layer": results_by_layer,
}

output_json_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"{EXPERIMENT_NAME}_full_results.json",
)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2, ensure_ascii=False)

print("Saved full Step 6 result JSON:")
print(output_json_path)

Saved full Step 6 result JSON:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step6_probe/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split_full_results.json


In [ ]:
zip_base = f"/content/{EXPERIMENT_NAME}_outputs"

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created Step 6 output zip:")
print(zip_path)

Created Step 6 output zip:
/content/step6_pilot_full_ithor120_diverse_qwen2_5_3b_composition_implicit_transitive_relation_ld_scene_split_outputs.zip


In [ ]:
from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>